# 1. Model Development and Evaluation

In [2]:
import pandas as pd

df = pd.read_csv("FINAL_newsprism_corpus.csv")

print(df.shape)
df.head()

(253, 13)


,article_id,event_id,topic,title,publisher,domain,url,published_at,language,source_country,raw_text,clean_text,word_count
0,ART_68BA904019A2,IMM_014,immigration,7NEWS Spotlight: Pauline Hanson’s UK tour and ...,7NEWS,7news.com.au,https://7news.com.au/spotlight/7news-spotlight...,2026-07-19T07:00:00Z,English,Australia,Pauline Hanson has rarely had a better opportu...,Pauline Hanson has rarely had a better opportu...,950
1,ART_041B7EA0CFF0,IMM_008,immigration,Australian man dies in immigration detention i...,7NEWS,7news.com.au,https://7news.com.au/news/australian-man-dies-...,2026-07-11T07:00:00Z,English,Australia,An Australian man has died in while in immigra...,An Australian man has died in while in immigra...,354
2,ART_21098433ADD2,IMM_004,immigration,Amanda Rishworth and Nat Barr clash over tradi...,7NEWS,7news.com.au,https://7news.com.au/sunrise/amanda-rishworth-...,2026-07-07T07:00:00Z,English,Australia,Employment Minister Amanda Rishworth has refus...,Employment Minister Amanda Rishworth has refus...,510
3,ART_C6E51BFB5116,IMM_007,immigration,Pauline Hanson meets with UK far-right figure ...,7NEWS,7news.com.au,https://7news.com.au/news/pauline-hanson-meets...,2026-07-09T07:00:00Z,English,Australia,Pauline Hanson has met with UK far-right figur...,Pauline Hanson has met with UK far-right figur...,338
4,ART_C956A4AFDAC0,IMM_025,immigration,The Issue Podcast: Demographer says the tax sy...,7NEWS,7news.com.au,https://7news.com.au/news/the-issue-podcast-de...,2026-08-08T07:00:00Z,English,Australia,Leading demographer Professor Peter McDonald s...,Leading demographer Professor Peter McDonald s...,931


In [3]:
# Create dataframe for sentiment

sentiment_df = df[[ "article_id","event_id", "topic", "publisher","title", "raw_text","clean_text"]].copy()
sentiment_df.head()

,article_id,event_id,topic,publisher,title,raw_text,clean_text
0,ART_68BA904019A2,IMM_014,immigration,7NEWS,7NEWS Spotlight: Pauline Hanson’s UK tour and ...,Pauline Hanson has rarely had a better opportu...,Pauline Hanson has rarely had a better opportu...
1,ART_041B7EA0CFF0,IMM_008,immigration,7NEWS,Australian man dies in immigration detention i...,An Australian man has died in while in immigra...,An Australian man has died in while in immigra...
2,ART_21098433ADD2,IMM_004,immigration,7NEWS,Amanda Rishworth and Nat Barr clash over tradi...,Employment Minister Amanda Rishworth has refus...,Employment Minister Amanda Rishworth has refus...
3,ART_C6E51BFB5116,IMM_007,immigration,7NEWS,Pauline Hanson meets with UK far-right figure ...,Pauline Hanson has met with UK far-right figur...,Pauline Hanson has met with UK far-right figur...
4,ART_C956A4AFDAC0,IMM_025,immigration,7NEWS,The Issue Podcast: Demographer says the tax sy...,Leading demographer Professor Peter McDonald s...,Leading demographer Professor Peter McDonald s...


In [ ]:
pip install transformers torch scikit-learn

# 2. RoBERTa Transformer

RoBERTa was selected as the final model for sentiment analysis due to its better overall performance compared to VADER. The complete workflow, including the model evaluation and selection process, is documented in sentiment_analysis_full.ipynb.

 This notebook focuses on applying RoBERTa to analyse headline and article sentiment, compare sentiment across news publishers and evaluate further language analysis for the NewsPrism application.

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import re

transformer = "cardiffnlp/twitter-roberta-base-sentiment-latest" # pretrained RoBERTa sentiment model

tokenizer = AutoTokenizer.from_pretrained(transformer) # Load the tokenizer

rob_model = AutoModelForSequenceClassification.from_pretrained(transformer)

rob_model.eval()

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [5]:
# Get the model's label IDs for each sentiment class
label_ids = {label.lower(): i
            for i, label in rob_model.config.id2label.items()}

NEG = label_ids["negative"]
NEU = label_ids["neutral"]
POS = label_ids["positive"]

The article text is split into smaller chunks of up to 500 tokens before being processed by RoBERTa. This is needed as RoBERTa has a maximum token limit and some articles may be too long to process at once. Splitting the articles allows the model to analyse the full article without losing information through truncation

### Analyse Article-level sentiment

In [6]:
misc = re.compile(r"(WATCH THE VIDEO[^.]*\.|"
    r"Know the news with the 7NEWS app[^.]*\.|"
    r"Stream free on.*$|"
    r"Sign up[^.]*newsletter[^.]*\.|"
    r"Listen to Australian and world news[^.]*\.)",
    re.I
)

SPLIT = re.compile(r'(?<=[.!?])\s+(?=[A-Z“‘"\'])')

def split_into_chunks(text, max_tokens=500):

    # Remove website noise
    text = misc.sub("", str(text))

    # Split article into sentences
    sentences = [sentence.strip() for sentence in SPLIT.split(text) if len(sentence.strip()) > 2]

    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:

        sentence_length = len(
            tokenizer.encode(
                sentence,
                add_special_tokens=False
            )
        )

        # Start a new chunk if adding the sentence
        if (
            current_chunk
            and current_length + sentence_length > max_tokens
        ):
            chunks.append(" ".join(current_chunk))

            current_chunk = []
            current_length = 0

        current_chunk.append(sentence)
        current_length += sentence_length

    # Add the final chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [7]:
def score_chunks(chunks, batch_size=16):

    results = []

    for i in range(0, len(chunks), batch_size):  # Process chunks in batches

        batch = chunks[i:i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )

        with torch.no_grad(): # Get sentiment probabilities without training the model

            logits = rob_model(**inputs).logits

            probabilities = torch.softmax( logits, dim=-1)

        for probability in probabilities:

            results.append({
                "negative": probability[NEG].item(),
                "neutral": probability[NEU].item(),
                "positive": probability[POS].item()
            })

    return results

In [8]:
def roberta_sentiment(text, max_tokens=500):

    # Split article into sentence chunks
    chunks = split_into_chunks(text, max_tokens=max_tokens)

    if not chunks:
        return pd.Series(["neutral", 0.0, 0.0, 0.0, 0.0, 0])

    # Get sentiment probabilities for each chunk
    chunk_scores = score_chunks(chunks)

    # Get token count of each chunk for weighting
    weights = [len(tokenizer.encode(chunk, add_special_tokens=False))
        for chunk in chunks
    ]

    total_weight = sum(weights)

    # Weighted average for each sentiment
    article_scores = {}

    for sentiment in ["negative", "neutral", "positive"]:
        article_scores[sentiment] = sum(
            score[sentiment] * weight
            for score, weight in zip(chunk_scores, weights)
        ) / total_weight

    # Sentiment with the highest probability as the final label
    label = max(article_scores, key=article_scores.get)
    confidence = article_scores[label]

    return pd.Series([
        label,
        confidence,
        article_scores["positive"],
        article_scores["neutral"],
        article_scores["negative"],
    ])

In [9]:
# Run model on all articles
sentiment_df = sentiment_df.copy()

sentiment_df[["article_roberta_label", "roberta_confidence", "roberta_positive", "roberta_neutral", "roberta_negative"]] = sentiment_df["raw_text"].apply(
    roberta_sentiment
)

sentiment_df[["title", "article_roberta_label", "roberta_confidence", "roberta_positive",  "roberta_neutral", "roberta_negative"]]

,title,article_roberta_label,roberta_confidence,roberta_positive,roberta_neutral,roberta_negative
0,7NEWS Spotlight: Pauline Hanson’s UK tour and ...,neutral,0.602124,0.146059,0.602124,0.251817
1,Australian man dies in immigration detention i...,negative,0.579514,0.011459,0.409027,0.579514
2,Amanda Rishworth and Nat Barr clash over tradi...,neutral,0.690928,0.032643,0.690928,0.276429
3,Pauline Hanson meets with UK far-right figure ...,neutral,0.737324,0.253893,0.737324,0.008784
4,The Issue Podcast: Demographer says the tax sy...,neutral,0.693434,0.031653,0.693434,0.274912
...,...,...,...,...,...,...
248,California and New York sue over Trump push to...,neutral,0.698762,0.035644,0.698762,0.265594
249,Dining across the divide: ‘He suggested the Cl...,neutral,0.589302,0.059848,0.589302,0.350850
250,‘Negligent’ to talk about long-term risks to A...,neutral,0.444766,0.132389,0.444766,0.422845
251,Australia’s dependence on gas generation durin...,neutral,0.697725,0.128466,0.697725,0.173809


In [10]:
# Count the number of positive, neutral and negative articles
sentiment_df["article_roberta_label"].value_counts()

,count
article_roberta_label,
neutral,208
negative,35
positive,10


### Analyse headline-level sentiment

In [11]:
def roberta_headline_sentiment(text):

    # Tokenize headline
    inputs = tokenizer(str(text),
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Get model prediction
    with torch.no_grad():
        logits = rob_model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    scores = {
        "negative": probabilities[NEG].item(),
        "neutral": probabilities[NEU].item(),
        "positive": probabilities[POS].item()
    }

    # Select sentiment with highest probability
    label = max(scores, key=scores.get)
    confidence = scores[label]

    return pd.Series([label, confidence])

In [12]:
sentiment_df[["headline_roberta_label", "headline_roberta_confidence"]] = sentiment_df["title"].apply(roberta_headline_sentiment)
sentiment_df[["title","headline_roberta_label","headline_roberta_confidence"]].head(10)

,title,headline_roberta_label,headline_roberta_confidence
0,7NEWS Spotlight: Pauline Hanson’s UK tour and ...,neutral,0.922676
1,Australian man dies in immigration detention i...,negative,0.698631
2,Amanda Rishworth and Nat Barr clash over tradi...,neutral,0.856469
3,Pauline Hanson meets with UK far-right figure ...,neutral,0.929275
4,The Issue Podcast: Demographer says the tax sy...,negative,0.503127
5,WA’s multibillion-dollar grain harvest at risk...,negative,0.648476
6,Labor immigration crackdown: Warning to foreig...,negative,0.498945
7,Prime Minister Anthony Albanese and Labor’s pl...,neutral,0.943889
8,"Military called in as more than 40,000 migrant...",neutral,0.934188
9,Home Affairs asks DV survivors to prove abuser...,neutral,0.921517


In [14]:
cols = ["title", "headline_roberta_label", "headline_roberta_confidence", "article_roberta_label", "roberta_confidence"]

display(sentiment_df[cols].head(10).style.format({"headline_roberta_confidence": "{:.2%}","article_roberta_confidence": "{:.2%}"}))

,title,headline_roberta_label,headline_roberta_confidence,article_roberta_label,roberta_confidence
0,7NEWS Spotlight: Pauline Hanson’s UK tour and meeting with Tommy Robinson exposes One Nation’s creative accounting on immigration numbers - 7NEWS,neutral,92.27%,neutral,0.602124
1,Australian man dies in immigration detention in Bali while awaiting deportation - 7NEWS,negative,69.86%,negative,0.579514
2,Amanda Rishworth and Nat Barr clash over tradie ranking in skilled migration scheme - 7NEWS,neutral,85.65%,neutral,0.690928
3,Pauline Hanson meets with UK far-right figure Tommy Robinson - 7NEWS,neutral,92.93%,neutral,0.737324
4,The Issue Podcast: Demographer says the tax system is ‘unfair’ for families amid the great population debate - 7NEWS,negative,50.31%,neutral,0.693434
5,WA’s multibillion-dollar grain harvest at risk as workers blocked or delayed - 7NEWS,negative,64.85%,negative,0.651031
6,"Labor immigration crackdown: Warning to foreign students, family visas and working holiday extensions - 7NEWS",negative,49.89%,neutral,0.653427
7,Prime Minister Anthony Albanese and Labor’s plan for reduced migration set to be announced before US trip - 7NEWS,neutral,94.39%,neutral,0.753094
8,"Military called in as more than 40,000 migrants from Morocco pour into Spanish border territory - ABC News & Headlines – Australian Broadcasting Corporation",neutral,93.42%,neutral,0.505387
9,Home Affairs asks DV survivors to prove abuser relationship 'genuine' - ABC News & Headlines – Australian Broadcasting Corporation,neutral,92.15%,neutral,0.506178


In [34]:
# Select headline and article sentiment results
sentiment_results = sentiment_df[
    ["article_id","publisher", "topic","title","headline_roberta_label", "headline_roberta_confidence","article_roberta_label", "roberta_confidence","roberta_positive",
      "roberta_neutral","roberta_negative"]].copy()

# Convert sentiment scores to percentages
cols = [ "headline_roberta_confidence", "roberta_confidence", "roberta_positive","roberta_neutral", "roberta_negative"]

sentiment_results[cols] = (sentiment_results[cols] * 100).round(1)

# Save combined results to csv as percentages
sentiment_results.to_csv("newsprism_sentiment_results.csv",index=False)

In [15]:
# Counts for headline sentiments
sentiment_df["headline_roberta_label"].value_counts()

,count
headline_roberta_label,
neutral,168
negative,72
positive,13


The results show that headline sentiment has greater variation than the sentiment of the full article content. This could be as headlines are short and designed to highlight the most important or attention-grabbing aspect of a story, whereas the article body gives more context and often uses more  factual and balanced language, resulting in more neutral sentiment overall.

In [16]:
# Check how much RoBERTa assigns the same sentiment to headlines and full articles
sentiment_df["roberta_headline_article_match"] = (sentiment_df["headline_roberta_label"] ==sentiment_df["article_roberta_label"])

In [17]:
# Checks how much the sentiment of the headline match the overall sentiment of the article
headline_article_match = (sentiment_df["roberta_headline_article_match"].mean()* 100)

print(f"Headline/article agreement: "
    f"{headline_article_match:.2f}%"
)

Headline/article agreement: 67.19%


### Publisher-Level Article Sentiment

Publisher-level analysis helps understand how the tone of news coverage differs across news sources. By comparing the distribution of positive, negative and neutral articles from each outlet, it can be observed whether certain sources tend to use more positive or negative sentiment in their reporting and identify patterns in how they cover news stories.

In [18]:
# Calculate article sentiment distribution for each news source
publisher_sent = pd.crosstab(
    sentiment_df["publisher"],
    sentiment_df["article_roberta_label"],
    normalize="index"
) * 100

publisher_sent = publisher_sent.round(1)
publisher_sent

article_roberta_label,negative,neutral,positive
publisher,,,
7NEWS,12.5,87.5,0.0
ABC News,10.9,84.8,4.3
News.com.au,21.1,78.9,0.0
SBS News,15.0,85.0,0.0
Sydney Morning Herald,11.3,81.1,7.5
The Age,11.1,83.3,5.6
The Guardian Australia,17.5,79.4,3.2


In [19]:
# Calculate headline sentiment distribution for each news source
publisher_sent = pd.crosstab(
    sentiment_df["publisher"],
    sentiment_df["headline_roberta_label"],
    normalize="index"
) * 100

publisher_sent = publisher_sent.round(1)
publisher_sent

headline_roberta_label,negative,neutral,positive
publisher,,,
7NEWS,31.2,68.8,0.0
ABC News,17.4,80.4,2.2
News.com.au,26.3,68.4,5.3
SBS News,25.0,70.0,5.0
Sydney Morning Herald,32.1,60.4,7.5
The Age,36.1,55.6,8.3
The Guardian Australia,30.2,65.1,4.8


In [20]:
# Calculate headline sentiment distribution for each publisher and topic
publisher_sent = pd.crosstab(
    [sentiment_df["topic"], sentiment_df["publisher"]],
    sentiment_df["headline_roberta_label"],
    normalize="index"
) * 100

# Arrange columns and round percentages
publisher_sent = publisher_sent.reindex(
    columns=["negative", "neutral", "positive"],
    fill_value=0
).round(1)

publisher_sent

headline_roberta_label               negative  neutral  positive
topic        publisher                                          
clean_energy 7NEWS                        0.0    100.0       0.0
             ABC News                    13.3     86.7       0.0
             News.com.au                100.0      0.0       0.0
             SBS News                    16.7     66.7      16.7
             Sydney Morning Herald       18.8     68.8      12.5
             The Age                     26.7     66.7       6.7
             The Guardian Australia      30.0     65.0       5.0
housing      7NEWS                       16.7     83.3       0.0
             ABC News                    12.5     87.5       0.0
             News.com.au                 15.4     76.9       7.7
             SBS News                    33.3     66.7       0.0
             Sydney Morning Herald       38.1     57.1       4.8
             The Age                     37.5     50.0      12.5
             The Guardian Australia      50.0     50.0       0.0
immigration  7NEWS                       50.0     50.0       0.0
             ABC News                    26.7     66.7       6.7
             News.com.au                 25.0     75.0       0.0
             SBS News                     0.0    100.0       0.0
             Sydney Morning Herald       37.5     56.2       6.2
             The Age                     60.0     40.0       0.0
             The Guardian Australia      20.0     73.3       6.7

In [20]:
# Example: Calculate negative sentiment percentages by publisher
negative_by_publisher = (sentiment_df.groupby("publisher").agg(negative_articles=(
            "article_roberta_label",
            lambda x: (x.str.lower() == "negative").mean() * 100
        ),
        negative_headlines=(
            "headline_roberta_label",
            lambda x: (x.str.lower() == "negative").mean() * 100
        )
    ).reset_index()
)

# Rename columns
negative_by_publisher.columns = ["Publisher", "Negative articles","Negative headlines"]

# Display as percentages
display(negative_by_publisher.style.format({ "Negative articles": "{:.1f}%", "Negative headlines": "{:.1f}%"}))

,Publisher,Negative articles,Negative headlines
0,7NEWS,12.5%,31.2%
1,ABC News,10.9%,17.4%
2,News.com.au,21.1%,26.3%
3,SBS News,15.0%,25.0%
4,Sydney Morning Herald,11.3%,32.1%
5,The Age,11.1%,36.1%
6,The Guardian Australia,17.5%,30.2%


In [36]:
# Save headline sentiment distribution to csv
publisher_sent.to_csv(
    "newsprism_publisher_headline_sentiment.csv",
    index=True
)

The results show that most news articles across all seven publishers were classified as neutral, showing that news outlets generally use more factual  language in their reporting. However, headlines were more likely to be negative, as they use stronger or more attention-grabbing language to attract readers.

News.com.au had the highest percentage of negative articles of 21.1%, while The Age had the highest percentage of negative headlines with 36.1%. Positive sentiment was relatively low across all publishers. Overall, this shows that while the articles themselves tend to have a neutral tone, headlines are more likely to use negative language. However, these differences depend on the political topics being covered and how RoBERTa interprets the text.

## Language Extraction

Following sentiment analysis, evaluative language extraction was carried out to better understand how news articles express positive or negative sentiment. While sentiment analysis identifies the overall tone of an article, language extraction helps identify the specific words that contribute to that tone. This allows us to explore how different news outlets use language when reporting on similar topics.

In [21]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import re, pandas as pd
from collections import Counter
import nltk

nltk.download("punkt_tab")

nltk.download('vader_lexicon')

lang_df = pd.read_csv("FINAL_newsprism_corpus.csv")
vader_lang = SentimentIntensityAnalyzer()

# News uses words like "party" or "support" neutrally, but VADER scores them as sentiment.
neutral_tone = ["party","support","super","wealth","values","clean","free","clear","ensure","kind","credits",
              "profit","negative","block","debt","argued","arguing","detention","ban","forced","fallen",
              "lowest","growth","benefit","good","great","better","united","won","parties","greater",
              "like","well","work","sure","important","ready","yes","no"]
for w in neutral_tone:
    vader_lang.lexicon[w] = 0.0

SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z“‘\"'])")
def sentences(text):
    return [s.strip() for s in SPLIT.split(str(text)) if len(s.strip()) > 2]

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [22]:
from nltk.tokenize import word_tokenize
from collections import Counter

def analyse(text, win=3, k=5, thresh=1.5):

    sents = sentences(text)

    neg_words = Counter()
    pos_words = Counter()
    sentence_scores = []

    for sent in sents:

        # Calculate sentiment score for each sentence
        score = vader_lang.polarity_scores(sent)["compound"]
        sentence_scores.append((sent, score))

       # Tokenise sentence into words
        words = [
            w.lower() for w in word_tokenize(sent)
            if w.isalpha()
        ]

        for word in words:

            # Get sentiment strength from VADER lexicon
            value = vader_lang.lexicon.get(word, 0)

            if abs(value) >= thresh:

                if value < 0:
                    neg_words[word] += 1
                else:
                    pos_words[word] += 1

    # Most positive and negative words
    if sentence_scores:
        worst = min(sentence_scores, key=lambda x: x[1])
        best = max(sentence_scores, key=lambda x: x[1])
    else:
        worst = ("", 0)
        best = ("", 0)

    return {
        "neg_words": [w for w, count in neg_words.most_common(k)],
        "pos_words": [w for w, count in pos_words.most_common(k)],
        "most_negative_sentence": worst[0] if worst[1] <= -0.05 else "",
        "most_positive_sentence": best[0] if best[1] >= 0.05 else ""
    }

In [23]:
# Apply sentiment analysis to all articles
emo = lang_df["raw_text"].apply(analyse).apply(pd.Series)

lang_df = pd.concat([lang_df, emo], axis=1)
lang_df = lang_df.loc[:, ~lang_df.columns.duplicated()].copy()

In [24]:
lang_df.columns
lang_df[["publisher", "topic", "title", "neg_words", "pos_words","most_negative_sentence","most_positive_sentence",]].head()

,publisher,topic,title,neg_words,pos_words,most_negative_sentence,most_positive_sentence
0,7NEWS,immigration,7NEWS Spotlight: Pauline Hanson’s UK tour and ...,"[illegal, lies, blames, war, sad]","[opportunity, special, popular, earnest, jewel]",WATCH THE VIDEO ABOVE: Hanson blames White Aus...,“This is a jewel of what you need to do in Aus...
1,7NEWS,immigration,Australian man dies in immigration detention i...,"[dead, died, emergency, attack, death]","[opportunities, help, trust]",“The results of the initial examination by the...,“Despite being given repeated opportunities to...
2,7NEWS,immigration,Amanda Rishworth and Nat Barr clash over tradi...,"[crisis, failing, rejected]",[],“What I would say is that when it comes to mak...,“We’ve had three times increase in the number ...
3,7NEWS,immigration,Pauline Hanson meets with UK far-right figure ...,"[fighting, criminal, denied, mistakes, destruc...","[true, fascinating, bravest, fearless, hopefully]",He has multiple criminal convictions and has p...,“Had a fascinating time chatting to one of the...
4,7NEWS,immigration,The Issue Podcast: Demographer says the tax sy...,"[unfair, discourages, denied, argues, problem]","[care, help, supporting, comfortable, popular]",He argues temporary migration is adding to Aus...,“The best thing a government could do to encou...


In [25]:
# Display full text
pd.set_option("display.max_colwidth", None)

# Display language analysis results for a random article
lang_df[["publisher","title","neg_words","pos_words"]].sample(1, random_state=42).T

,208
publisher,ABC News
title,How a controversial scheme will fund Australia's largest koala park - abc.net.au
neg_words,"[rejected, polluters, fault, lies, banned]"
pos_words,"[credit, encourage, benefits, dedicated, help]"


### Display only top 3 negative and positive words for each article

Only top 3 negative and positive words were selected to display in the streamlit UI, to reduce clutter and make it more visually appealing.

In [26]:
# Select columns for all articles
language_results = lang_df[["article_id", "topic", "publisher", "title", "neg_words", "pos_words"]].copy()

# Keep only the top 3 positive and negative words
for col in ["neg_words", "pos_words"]:
    language_results[col] = language_results[col].apply(
        lambda words: words[:3]
    )

display(language_results)


,article_id,topic,publisher,title,neg_words,pos_words
0,ART_68BA904019A2,immigration,7NEWS,7NEWS Spotlight: Pauline Hanson’s UK tour and meeting with Tommy Robinson exposes One Nation’s creative accounting on immigration numbers - 7NEWS,"[illegal, lies, blames]","[opportunity, special, popular]"
1,ART_041B7EA0CFF0,immigration,7NEWS,Australian man dies in immigration detention in Bali while awaiting deportation - 7NEWS,"[dead, died, emergency]","[opportunities, help, trust]"
2,ART_21098433ADD2,immigration,7NEWS,Amanda Rishworth and Nat Barr clash over tradie ranking in skilled migration scheme - 7NEWS,"[crisis, failing, rejected]",[]
3,ART_C6E51BFB5116,immigration,7NEWS,Pauline Hanson meets with UK far-right figure Tommy Robinson - 7NEWS,"[fighting, criminal, denied]","[true, fascinating, bravest]"
4,ART_C956A4AFDAC0,immigration,7NEWS,The Issue Podcast: Demographer says the tax system is ‘unfair’ for families amid the great population debate - 7NEWS,"[unfair, discourages, denied]","[care, help, supporting]"
...,...,...,...,...,...,...
248,ART_2D6D9C623FC9,clean_energy,The Guardian Australia,California and New York sue over Trump push to halt wind energy projects - The Guardian,"[illegal, discourage, rob]","[hand, favor, confident]"
249,ART_F147CC01621A,clean_energy,The Guardian Australia,Dining across the divide: ‘He suggested the Clacton byelection was a vanity project. Obviously I disagree’ - The Guardian,"[abuse, disagree, damage]","[nice, amuse, easy]"
250,ART_A531A7AC7600,clean_energy,The Guardian Australia,"‘Negligent’ to talk about long-term risks to Australia without examining cost of climate crisis, experts say - The Guardian","[damage, severe, criticised]","[greatest, natural, clearly]"
251,ART_B8D83C2A515B,clean_energy,The Guardian Australia,Australia’s dependence on gas generation during evening energy peak falls by almost 70% - The Guardian,[problem],"[asset, win, profitable]"


In [27]:
# Save language extraction results to csv
language_results.to_csv(
    "newsprism_language_results.csv",
    index=False
)

The results show that language extraction identifies positive and negative words and sentences within each news article. For example, negative words such as crisis, illegal, rejected and failing were detected, while positive words included opportunity, support, benefit and better. The analysis also identifies the sentences with the strongest positive and negative sentiment scores.

However, some extracted words and sentences may not accurately reflect the overall tone of the reporting, as they can describe events or quote people's opinions rather than express the journalist's own views. This highlights a limitation of lexicon-based language extraction, where words are not always interpreted in their full context.

